# GeoLine — versão interativa para Google Colab

Este notebook oferece uma interface no navegador para construir retas no R² e no R³, analisar a relação entre duas retas e exibir gráficos. Os cálculos são executados pelas classes originais do GeoLine.

> Execute as células na ordem. A célula de clonagem baixa a versão atual da branch `main` do repositório GeoLine.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORIO = "https://github.com/igordesouzaDev/GeoLine.git"
BRANCH_GEOLINE = "main"
PASTA_GEOLINE = Path("/content/GeoLine")

if not PASTA_GEOLINE.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH_GEOLINE,
         REPOSITORIO, str(PASTA_GEOLINE)],
        check=True,
    )
elif not (PASTA_GEOLINE / ".git").exists():
    raise RuntimeError(f"Já existe {PASTA_GEOLINE}, mas ela não é um clone do GeoLine.")

os.chdir(PASTA_GEOLINE)
if str(PASTA_GEOLINE) not in sys.path:
    sys.path.insert(0, str(PASTA_GEOLINE))

print(f"GeoLine carregado de: {PASTA_GEOLINE}")

## Dependências

O backend matemático usa apenas recursos nativos do Python. Para a interface neste notebook, instalamos `ipywidgets`; os gráficos usam Matplotlib, já declarado no `requirements.txt` do projeto.

In [ ]:
%pip -q install -r requirements.txt ipywidgets

In [ ]:
from IPython.display import Markdown, clear_output, display
import ipywidgets as widgets
from matplotlib.backends.backend_agg import FigureCanvasAgg

from backend.analisador import Analisador
from backend.reta_r2 import RetaR2
from backend.reta_r3 import RetaR3
from backend.vetor import Vetor
from visualizacao.graficos import Graficos

print("Classes do GeoLine importadas com sucesso.")

## Interface

Selecione a operação e a dimensão. Na construção, informe os dois pontos da reta. Na análise, informe os pontos de duas retas. Na visualização, escolha uma ou duas retas. Os exemplos preenchidos podem ser substituídos.

In [ ]:
def _ler_ponto(campos_ponto, dimensao, descricao):
    coordenadas = []
    for campo in campos_ponto[:dimensao]:
        texto = campo.value.strip().replace(",", ".")
        if not texto:
            raise ValueError(f"Preencha todas as coordenadas de {descricao}.")
        try:
            coordenadas.append(float(texto))
        except ValueError as erro:
            raise ValueError(f"Informe um número válido em {descricao}.") from erro
    return Vetor(coordenadas)


def _ler_reta(numero, dimensao):
    campos_reta = campos_retas[numero]
    inicial = _ler_ponto(campos_reta["inicial"], dimensao,
                         f"Reta {numero}, ponto inicial")
    final = _ler_ponto(campos_reta["final"], dimensao,
                       f"Reta {numero}, ponto final")
    classe = RetaR2 if dimensao == 2 else RetaR3
    return classe(inicial, final)


def _mostrar_equacoes(reta, dimensao):
    display(Markdown(f"### Reta no R{dimensao}"))
    display(Markdown(
        f"- **Ponto inicial:** `{reta.ponto_inicial}`  \n"
        f"- **Ponto final:** `{reta.obter_ponto(1)}`  \n"
        f"- **Vetor diretor:** `{reta.vetor_diretor}`  \n"
        f"- **Equação vetorial:** `{reta.equacao_vetorial()}`  \n"
        f"- **Equações paramétricas:**  \n```text\n{reta.equacao_parametrica()}\n```"
    ))
    display(Markdown(f"- **Equação simétrica:** `{reta.equacao_simetrica()}`"))
    if dimensao == 2:
        display(Markdown(f"- **Equação reduzida:** `{reta.equacao_reduzida()}`"))


def _mostrar_analise(reta1, reta2, dimensao):
    resultado = Analisador.analisar(reta1, reta2)
    display(Markdown(f"### Relação: {resultado['relacao'].capitalize()}"))
    display(Markdown(
        f"{resultado['mensagem']}  \n"
        f"- Paralelas: **{'Sim' if resultado['sao_paralelas'] else 'Não'}**  \n"
        f"- Coincidentes: **{'Sim' if resultado['sao_coincidentes'] else 'Não'}**  \n"
        f"- Reversas: **{'Sim' if resultado['sao_reversas'] else 'Não'}**  \n"
        f"- Interseção: **{resultado['ponto_intersecao'] or 'não há ponto único'}**"
    ))
    if dimensao == 3:
        produto_escalar = reta1.vetor_diretor.produto_escalar(reta2.vetor_diretor)
        display(Markdown(
            f"- Produto escalar dos vetores diretores: **{produto_escalar:g}**  \n"
            f"- Ortogonais: **{'Sim' if reta1.eh_ortogonal(reta2) else 'Não'}**  \n"
            f"- Ângulo entre as retas: **{reta1.angulo_com(reta2):g}°**  \n"
            f"- Distância entre as retas: **{reta1.distancia_reta(reta2):g}**"
        ))
    return resultado


def _mostrar_grafico(retas, dimensao):
    if dimensao == 2:
        figura, _ = (Graficos.plotar_reta_r2(retas[0]) if len(retas) == 1
                     else Graficos.plotar_duas_retas_r2(*retas))
    else:
        figura, _ = (Graficos.plotar_reta_r3(retas[0]) if len(retas) == 1
                     else Graficos.plotar_duas_retas_r3(*retas))
    display(figura)
    return figura


In [ ]:
operacao = widgets.Dropdown(
    options=["Construir reta", "Analisar duas retas", "Visualizar gráfico"],
    value="Construir reta",
    description="Operação:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="380px"),
)
dimensao = widgets.ToggleButtons(
    options=[("R²", 2), ("R³", 3)], value=2, description="Espaço:",
)
quantidade = widgets.ToggleButtons(
    options=[("Uma reta", 1), ("Duas retas", 2)], value=1,
    description="Quantidade:",
)
grupo_quantidade = widgets.HBox([quantidade])

campos_retas = {}
paineis_retas = {}
for numero in (1, 2):
    campos_retas[numero] = {}
    grupos_pontos = []
    for nome_ponto, rotulo in (("inicial", "Ponto inicial"), ("final", "Ponto final")):
        coordenadas = [
            widgets.Text(value=valor, description=eixo,
                         layout=widgets.Layout(width="175px"),
                         style={"description_width": "25px"})
            for eixo, valor in zip(("X", "Y", "Z"), ("0", "0", "0"))
        ]
        campos_retas[numero][nome_ponto] = coordenadas
        grupos_pontos.append(widgets.VBox([
            widgets.HTML(f"<b>{rotulo}</b>"), widgets.HBox(coordenadas)
        ], layout=widgets.Layout(margin="4px 0 10px 0")))
    paineis_retas[numero] = widgets.VBox(
        [widgets.HTML(f"<h4>Reta {numero}</h4>"), *grupos_pontos],
        layout=widgets.Layout(border="1px solid #DCE3EC", padding="10px",
                              margin="6px 0", width="100%"),
    )

# Dois exemplos diferentes facilitam visualizar duas retas no mesmo gráfico.
for eixo, valor in zip(campos_retas[1]["final"], ("3", "2", "1")):
    eixo.value = valor
for eixo, valor in zip(campos_retas[2]["inicial"], ("0", "2", "1")):
    eixo.value = valor
for eixo, valor in zip(campos_retas[2]["final"], ("3", "0", "2")):
    eixo.value = valor

formulario = widgets.VBox([paineis_retas[1], paineis_retas[2]])
botao_executar = widgets.Button(
    description="Executar", button_style="primary",
    icon="play", layout=widgets.Layout(width="180px", height="42px"),
)
saida = widgets.Output(layout=widgets.Layout(
    border="1px solid #DCE3EC", padding="12px", width="100%"
))


def _atualizar_controles(_mudanca=None):
    numero_retas = 2 if operacao.value == "Analisar duas retas" else quantidade.value
    paineis_retas[1].layout.display = ""
    paineis_retas[2].layout.display = "" if numero_retas == 2 else "none"
    grupo_quantidade.layout.display = "" if operacao.value == "Visualizar gráfico" else "none"
    for campos_reta in campos_retas.values():
        for campos_ponto in campos_reta.values():
            campos_ponto[2].layout.display = "" if dimensao.value == 3 else "none"


operacao.observe(_atualizar_controles, names="value")
dimensao.observe(_atualizar_controles, names="value")
quantidade.observe(_atualizar_controles, names="value")
_atualizar_controles()


def _executar(_botao):
    with saida:
        clear_output(wait=True)
        try:
            espaco = dimensao.value
            if operacao.value == "Analisar duas retas":
                reta1, reta2 = _ler_reta(1, espaco), _ler_reta(2, espaco)
                _mostrar_analise(reta1, reta2, espaco)
            elif operacao.value == "Construir reta":
                reta = _ler_reta(1, espaco)
                _mostrar_equacoes(reta, espaco)
            else:
                retas = [_ler_reta(numero, espaco)
                         for numero in range(1, quantidade.value + 1)]
                _mostrar_grafico(retas, espaco)
        except (TypeError, ValueError, OverflowError, RuntimeError) as erro:
            display(Markdown(f"**Não foi possível executar:** {erro}"))


botao_executar.on_click(_executar)
interface = widgets.VBox([
    widgets.HTML("<h2>GeoLine</h2><p>Construção, análise e visualização de retas.</p>"),
    widgets.HBox([operacao, dimensao]),
    grupo_quantidade,
    formulario,
    botao_executar,
    saida,
], layout=widgets.Layout(width="100%", margin="6px 0"))
display(interface)

## Verificações da versão Colab

A célula seguinte exercita os construtores e métodos do backend, as relações geométricas, os quatro tipos de gráfico e o estado dos controles. Ela também simula o clique do botão da interface. A execução comprova os fluxos em um kernel Jupyter compatível; a disponibilidade deste notebook no Google Colab ainda depende de ele ser publicado no GitHub.

In [ ]:
# Construção de retas e equações nos dois espaços.
reta2 = RetaR2(Vetor([0, 0]), Vetor([2, 4]))
assert reta2.vetor_diretor.coordenadas == (2, 4)
assert "x =" in reta2.equacao_parametrica()
assert reta2.equacao_simetrica()
assert reta2.equacao_reduzida()

reta3 = RetaR3(Vetor([0, 0, 0]), Vetor([2, 4, 6]))
assert reta3.vetor_diretor.coordenadas == (2, 4, 6)
assert "z =" in reta3.equacao_parametrica()
assert reta3.equacao_simetrica()

# O analisador original classifica interseção, paralelismo, coincidência e reversidade.
concorrente_r2 = RetaR2(Vetor([0, 2]), Vetor([2, 0]))
assert Analisador.analisar(reta2, concorrente_r2)["relacao"] == "concorrentes"
paralela_r2 = RetaR2(Vetor([0, 1]), Vetor([2, 5]))
assert Analisador.analisar(reta2, paralela_r2)["relacao"] == "paralelas"
coincidente_r2 = RetaR2(Vetor([1, 2]), Vetor([3, 6]))
assert Analisador.analisar(reta2, coincidente_r2)["relacao"] == "coincidentes"

reta_x = RetaR3(Vetor([0, 0, 0]), Vetor([1, 0, 0]))
reta_skew = RetaR3(Vetor([0, 1, 1]), Vetor([0, 2, 1]))
assert Analisador.analisar(reta_x, reta_skew)["relacao"] == "reversas"
reta_y = RetaR3(Vetor([1, -1, 0]), Vetor([1, 1, 0]))
assert Analisador.analisar(reta_x, reta_y)["relacao"] == "concorrentes"

# As figuras podem ser renderizadas sem Tkinter, usando o backend Agg do Matplotlib.
casos_graficos = [
    (Graficos.plotar_reta_r2(reta2), 2),
    (Graficos.plotar_duas_retas_r2(reta2, concorrente_r2), 2),
    (Graficos.plotar_reta_r3(reta3), 3),
    (Graficos.plotar_duas_retas_r3(reta_x, reta_skew), 3),
]
for (figura, eixo), dimensao_grafico in casos_graficos:
    total_retas = 2 if len(eixo.get_legend().get_texts()) == 2 else 1
    esperado = total_retas + (2 if dimensao_grafico == 2 else 0)
    assert len(eixo.lines) == esperado
    FigureCanvasAgg(figura).draw()
    figura.clear()

# Entradas inválidas continuam sendo rejeitadas pelas classes do projeto.
try:
    RetaR2(Vetor([1, 1]), Vetor([1, 1]))
except ValueError:
    pass
else:
    raise AssertionError("A reta com pontos iguais deveria ser rejeitada.")

try:
    Vetor([float("inf"), 0])
except ValueError:
    pass
else:
    raise AssertionError("A coordenada infinita deveria ser rejeitada.")

# Os campos do notebook também rejeitam valores vazios e texto não numérico.
campos_teste = [widgets.Text(value=""), widgets.Text(value="2")]
try:
    _ler_ponto(campos_teste, 2, "ponto de teste")
except ValueError:
    pass
else:
    raise AssertionError("Um campo vazio deveria ser rejeitado.")
campos_teste[0].value = "abc"
try:
    _ler_ponto(campos_teste, 2, "ponto de teste")
except ValueError:
    pass
else:
    raise AssertionError("Um valor não numérico deveria ser rejeitado.")

# Verifica a reconstrução/visibilidade dos controles para as opções da interface.
operacao.value = "Analisar duas retas"
assert paineis_retas[2].layout.display == ""
operacao.value = "Visualizar gráfico"
quantidade.value = 2
assert paineis_retas[2].layout.display == ""
dimensao.value = 2
assert campos_retas[1]["inicial"][2].layout.display == "none"
dimensao.value = 3
assert campos_retas[1]["inicial"][2].layout.display == ""

# Verifica o callback registrado e a conversão da entrada da interface em uma reta.
operacao.value = "Construir reta"
dimensao.value = 2
quantidade.value = 1
campos_retas[1]["inicial"][0].value = "0"
campos_retas[1]["inicial"][1].value = "0"
campos_retas[1]["final"][0].value = "2"
campos_retas[1]["final"][1].value = "4"
reta_da_interface = _ler_reta(1, 2)
assert reta_da_interface.vetor_diretor.coordenadas == (2, 4)
assert botao_executar._click_handlers.callbacks

print("Verificações concluídas: retas R²/R³, análise, entradas inválidas, controles e gráficos 2D/3D.")

## Como compartilhar

Depois que esta branch for enviada ao GitHub, o notebook poderá ser aberto diretamente no Colab pelo endereço:

`https://colab.research.google.com/github/igordesouzaDev/GeoLine/blob/feature/versao-google-colab/notebooks/GeoLine_Colab.ipynb`

Para compartilhar com o professor, envie esse endereço depois que o arquivo estiver publicado. O repositório precisa estar acessível para quem abrir o link.